# Optimal control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/tutorial/advanced/02_optimal_control.ipynb)

One pendulum swing-up, four planners: LQR, value iteration, trajectory optimization and RRT, compared on the same run.

In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")

## 1. One problem

The task: swing a pendulum from rest at the bottom, $x = (0, 0)$, to the upright,
$x^\star = (\pi, 0)$, with a motor limited to $|u| \le 5$ (the torque gravity needs to hold
the pendulum horizontal is $mgl \approx 9.8$: it must pump energy). The cost is

$$J = \int_0^{t_f} (x - x^\star)^T Q\, (x - x^\star) + u^T R\, u \; dt,$$

the state stays in a box `X` (the grid of value iteration and the tree of RRT need one),
and the goal `Xf` is a ball of radius 0.3 around the upright.

In [ ]:
import numpy as np

from minilink import BallSet, Pendulum, PlanningProblem, QuadraticCost

pendulum = Pendulum()
pendulum.state.lower_bound = np.array([-2.0 * np.pi, -8.0])
pendulum.state.upper_bound = np.array([2.0 * np.pi, 8.0])

x_up = np.array([np.pi, 0.0])
cost = QuadraticCost.from_system(pendulum, x_bar=x_up, Q=np.eye(2), R=0.1 * np.eye(1))
problem = PlanningProblem(
    pendulum,
    x_start=np.zeros(2),
    cost=cost,
    X=pendulum.state.box,
    U=pendulum.input_box,
    Xf=BallSet(x_up, 0.3),
    horizon=5.0,
)
problem

## 2. Four planners

Each planner reads the same problem and returns a `PlanningSolution`; `evaluate=True`
also rolls its policy out on the plant from the start. They differ in what they return.

**LQR** linearizes at the target and solves the Riccati equation: a feedback law
$u = -K(t)(x - x^\star)$, exact near the upright, blind to `U`.

In [ ]:
from minilink import LQRPlanner

lqr = LQRPlanner(problem).solve(evaluate=True)

**Value iteration** grids the state space (61 × 61 points) and the input (11 torques),
and iterates the Bellman equation on the discretized dynamics until the cost-to-go stops
changing:

$$J(x) = \min_{u \in U} \big[\, g(x, u)\,\Delta t + J(x + f(x, u)\,\Delta t) \,\big].$$

Its policy is a lookup table over the grid: a global feedback law.

In [ ]:
from minilink import DynamicProgrammingPlanner

vi_planner = DynamicProgrammingPlanner(
    problem, x_grid=(61, 61), u_grid=(11,), dt=0.05, tol=0.1, max_iterations=3000
)
vi = vi_planner.solve(evaluate=True)

**Trajectory optimization** transcribes the problem into a finite optimization over 25
time samples (direct collocation). It needs a point target, so it solves the same problem
with `Xf` replaced by the upright state. Its answer is an open-loop plan $u(t)$.

In [ ]:
from minilink import TrajectoryOptimizationPlanner

trajopt_planner = TrajectoryOptimizationPlanner(
    problem.replace(Xf=x_up), n_steps=25, transcription="direct_collocation"
)
trajopt = trajopt_planner.solve(evaluate=True)

**RRT** grows a tree of feasible motions from the start: each extension applies one of
five torques for 0.3 s, until a node lands in `Xf`. Its answer is also an open-loop plan,
feasible but not optimal.

In [ ]:
from minilink import RRTPlanner
from minilink.planning.search.extenders import KinodynamicExtender
from minilink.planning.search.rrt import RRTOptions

torques = [np.array([tau]) for tau in (-5.0, -2.0, 0.0, 2.0, 5.0)]
extender = KinodynamicExtender(controls=torques, horizon=0.3, n_substeps=6)
rrt_planner = RRTPlanner(
    problem, extender=extender, options=RRTOptions(seed=0, max_nodes=20000)
)
rrt = rrt_planner.solve(evaluate=True)

## 3. Compared on the same run

`compare` scores every answer on the same simulation of the problem: its cost $J$, whether
it respects `U` (the feasibility margin, negative when violated) and whether it ends in
`Xf`.

In [ ]:
from minilink import compare

comparison = compare(problem, LQR=lqr, VI=vi, TrajOpt=trajopt, RRT=rrt, dt=0.01)
print(comparison)

How to read the table:

- **LQR** goes straight up with far more torque than the motor has (a margin of about
  $-58$): it is a local law, and here it is used far from its operating point.
- **Value iteration** swings up and ends in `Xf`, with the lowest cost. Its "failed" is a
  known rounding defect: the lookup policy interpolates between grid torques and can exceed
  the bound by about $10^{-15}$, which the strict test against `U` counts as a violation.
- **Trajectory optimization** has the optimal plan on its own model ($J^\star$ in the solver
  column), and **RRT** a feasible one. Both are open loop: replayed on the plant, the small
  integration differences grow near the unstable upright and the pendulum falls past the
  goal. A plan needs a feedback loop around it; that is model predictive control, in the
  next notebook.

The overlay shows each answer's own trajectory: the rollout of each feedback law, and the
plan of each open-loop planner as the planner computed it. The plans reach the upright;
their replay on the plant, scored in the table, does not.

In [ ]:
comparison.plot_trajectory()

## 4. What each planner returns

The two feedback laws can be drawn over the state space. Value iteration's law is nearly
bang-bang, full torque one way or the other, in bands that pump energy into the swing; LQR's
is a plane $u = -K(x - x^\star)$, beyond the motor's limits nearly everywhere (the color
scale clips it at the bounds).

In [ ]:
feedback_laws = compare(VI=vi, LQR=lqr)
feedback_laws.plot_control_law()

Value iteration also returns its cost-to-go $J(x)$ over the whole grid, the price of
reaching the upright from every state (the color scale stops at 100; the states from which the
pendulum leaves the box cost far more).

In [ ]:
vi_planner.plot_cost_to_go(jmax=100.0)

RRT keeps its tree: every motion it tried, and the branch that reached the goal.

In [ ]:
rrt_planner.plot_tree(x_axis=0, y_axis=1)

## Summary

| Planner | Returns | Uses `U` | Global | Cost |
| --- | --- | --- | --- | --- |
| `LQRPlanner` | feedback $u = -K(t)(x - x^\star)$ | no | no | optimal near $x^\star$ |
| `DynamicProgrammingPlanner` | feedback lookup table, cost-to-go | yes | yes | optimal on the grid |
| `TrajectoryOptimizationPlanner` | open-loop plan $u(t)$ | yes | from its start | locally optimal |
| `RRTPlanner` | open-loop plan $u(t)$, tree | yes | from its start | feasible only |

Next, [`03_mpc_and_mppi`](03_mpc_and_mppi.ipynb): closing the loop around a planner.